# Parte 1: Planificación Automática para Gestión Inteligente de un Almacén Autónomo
## RAAP - Ruta de Aprendizaje Autónomo Parte 1

**Empresa:** AutoLogistics
**Objetivo:** Construir un planificador automático para transportar cajas entre zonas de almacén

**Problema:** El robot debe llevar una caja pesada (requiere carretilla) desde Zona A a Zona C, pero:
- La carretilla está lejos (en almacén)
- Zona C está bloqueada
- Necesita una secuencia de acciones coherente

## Imports y Configuración

In [ ]:
from collections import deque
import heapq
import time

print("✓ Librerías cargadas")

## Tarea 1: Definición del Dominio y Problema PDDL

### Clase Estado
Representa el estado del problema usando un conjunto de predicados.

In [ ]:
class Estado:
    """Representa el estado del problema como predicados lógicos."""
    
    def __init__(self, robot_zona, caja_zona, carretilla_zona, tiene_carretilla, robot_cargado, zona_c_bloqueada):
        self.robot_zona = robot_zona
        self.caja_zona = caja_zona
        self.carretilla_zona = carretilla_zona
        self.tiene_carretilla = tiene_carretilla
        self.robot_cargado = robot_cargado
        self.zona_c_bloqueada = zona_c_bloqueada
    
    def __eq__(self, otro):
        return (self.robot_zona == otro.robot_zona and 
                self.caja_zona == otro.caja_zona and
                self.carretilla_zona == otro.carretilla_zona and
                self.tiene_carretilla == otro.tiene_carretilla and
                self.robot_cargado == otro.robot_cargado and
                self.zona_c_bloqueada == otro.zona_c_bloqueada)
    
    def __hash__(self):
        return hash((self.robot_zona, self.caja_zona, self.carretilla_zona, 
                    self.tiene_carretilla, self.robot_cargado, self.zona_c_bloqueada))
    
    def __repr__(self):
        cargo = f" (cargando {self.robot_cargado})" if self.robot_cargado else ""
        return f"Estado(robot:{self.robot_zona}, caja:{self.caja_zona}{cargo})"

print("✓ Clase Estado definida")

### Estado Inicial y Objetivo

In [ ]:
# Estado inicial
estado_inicial = Estado(
    robot_zona="A",
    caja_zona="A",
    carretilla_zona="almacen",
    tiene_carretilla=False,
    robot_cargado=None,
    zona_c_bloqueada=True
)

print("Estado inicial:")
print(f"  • Robot en: Zona A")
print(f"  • Caja (pesada) en: Zona A")
print(f"  • Carretilla en: Almacén")
print(f"  • Zona C: Bloqueada")
print(f"\nObjetivo: Caja en Zona C")

### Generador de Acciones
Define las 5 acciones STRIPS del dominio.

In [ ]:
def generar_acciones(estado):
    """Genera todas las acciones aplicables en el estado actual."""
    acciones = []
    zonas = ["A", "B", "C", "almacen"]
    
    # 1. Mover robot a zona adyacente
    movimientos = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B"), ("A", "almacen"), ("almacen", "A")]
    for origen, destino in movimientos:
        if estado.robot_zona == origen:
            if destino == "C" and estado.zona_c_bloqueada:
                continue
            nuevo_estado = Estado(destino, estado.caja_zona, estado.carretilla_zona, 
                                estado.tiene_carretilla, estado.robot_cargado, estado.zona_c_bloqueada)
            acciones.append((f"Mover robot de {origen} a {destino}", nuevo_estado))
    
    # 2. Buscar carretilla
    if (estado.robot_zona == estado.carretilla_zona and not estado.tiene_carretilla):
        nuevo_estado = Estado(estado.robot_zona, estado.caja_zona, estado.carretilla_zona,
                            True, estado.robot_cargado, estado.zona_c_bloqueada)
        acciones.append((f"Tomar carretilla en {estado.robot_zona}", nuevo_estado))
    
    # 3. Recoger caja (necesita carretilla porque es pesada)
    if (estado.robot_zona == estado.caja_zona and 
        estado.robot_cargado is None and
        estado.tiene_carretilla):
        nuevo_estado = Estado(estado.robot_zona, None, estado.carretilla_zona,
                            estado.tiene_carretilla, "caja", estado.zona_c_bloqueada)
        acciones.append((f"Recoger caja en {estado.robot_zona} (con carretilla)", nuevo_estado))
    
    # 4. Depositar caja
    if estado.robot_cargado == "caja":
        nuevo_estado = Estado(estado.robot_zona, estado.robot_zona, estado.carretilla_zona,
                            estado.tiene_carretilla, None, estado.zona_c_bloqueada)
        acciones.append((f"Depositar caja en {estado.robot_zona}", nuevo_estado))
    
    # 5. Despejar zona C
    if estado.zona_c_bloqueada:
        nuevo_estado = Estado(estado.robot_zona, estado.caja_zona, estado.carretilla_zona,
                            estado.tiene_carretilla, estado.robot_cargado, False)
        acciones.append((f"Despejar zona C", nuevo_estado))
    
    return acciones

print("✓ Generador de acciones definido (5 acciones STRIPS)")

## Tarea 2: Implementación de Planificadores

### Búsqueda BFS (Breadth-First Search)

In [ ]:
def busqueda_bfs(estado_inicial):
    """Búsqueda en amplitud - garantiza plan óptimo."""
    cola = deque([(estado_inicial, [])])
    visitados = {estado_inicial}
    iteraciones = 0
    
    while cola and iteraciones < 10000:
        estado_actual, plan = cola.popleft()
        iteraciones += 1
        
        # Verificar objetivo
        if estado_actual.caja_zona == "C":
            return plan, iteraciones, True
        
        acciones = generar_acciones(estado_actual)
        for nombre, nuevo_estado in acciones:
            if nuevo_estado not in visitados:
                visitados.add(nuevo_estado)
                nuevo_plan = plan + [nombre]
                cola.append((nuevo_estado, nuevo_plan))
    
    return [], iteraciones, False

print("✓ BFS implementado")

### Búsqueda A* (con heurística)

In [ ]:
def heuristica(estado):
    """Heurística simple para guiar búsqueda A*."""
    if estado.caja_zona == "C":
        return 0
    
    h = 0
    if not estado.tiene_carretilla:
        h += 2  # Ir a almacén + tomar carretilla
    if estado.robot_cargado != "caja":
        h += 1  # Recoger caja
    if estado.zona_c_bloqueada:
        h += 1  # Despejar zona C
    if estado.caja_zona != "C":
        h += 2  # Mover a zona C
    return h

def busqueda_astar(estado_inicial):
    """Búsqueda A* - combina optimalidad con eficiencia."""
    cola = [(heuristica(estado_inicial), 0, estado_inicial, [])]
    visitados = {estado_inicial}
    iteraciones = 0
    contador = 0
    
    while cola and iteraciones < 10000:
        _, _, estado_actual, plan = heapq.heappop(cola)
        iteraciones += 1
        
        if estado_actual.caja_zona == "C":
            return plan, iteraciones, True
        
        acciones = generar_acciones(estado_actual)
        for nombre, nuevo_estado in acciones:
            if nuevo_estado not in visitados:
                visitados.add(nuevo_estado)
                nuevo_plan = plan + [nombre]
                g = len(nuevo_plan)
                h = heuristica(nuevo_estado)
                f = g + h
                heapq.heappush(cola, (f, contador, nuevo_estado, nuevo_plan))
                contador += 1
    
    return [], iteraciones, False

print("✓ A* implementado")

In [ ]:
print("\n" + "="*80)
print("VERIFICACIÓN DEL PLAN A*")
print("="*80)

if encontrado_astar and plan_astar:
    print(f"\n✅ Plan encontrado correctamente")
    print(f"   Longitud: {len(plan_astar)} acciones")
    print(f"   Iteraciones exploradas: {iter_astar}")
    print(f"   Tiempo: {tiempo_astar:.6f}s")
    print(f"\n   Es ÓPTIMO porque:")
    print(f"   • A* con heurística admisible garantiza optimalidad")
    print(f"   • No hay forma de hacerlo en menos pasos")

## Ejecución y Resultados

In [ ]:
print("="*80)
print("BÚSQUEDA BFS")
print("="*80)

inicio = time.time()
plan_bfs, iter_bfs, encontrado_bfs = busqueda_bfs(estado_inicial)
tiempo_bfs = time.time() - inicio

print(f"\n✓ Búsqueda completada en {tiempo_bfs:.6f} segundos")
print(f"  • Iteraciones: {iter_bfs}")
print(f"  • Plan encontrado: {'Sí ✓' if encontrado_bfs else 'No'}")
print(f"  • Longitud: {len(plan_bfs)} acciones")

if encontrado_bfs:
    print(f"\n📝 PLAN:")
    for i, accion in enumerate(plan_bfs, 1):
        print(f"  {i}. {accion}")

In [ ]:
print("\n" + "="*80)
print("BÚSQUEDA A*")
print("="*80)

inicio = time.time()
plan_astar, iter_astar, encontrado_astar = busqueda_astar(estado_inicial)
tiempo_astar = time.time() - inicio

print(f"\n✓ Búsqueda completada en {tiempo_astar:.6f} segundos")
print(f"  • Iteraciones: {iter_astar}")
print(f"  • Plan encontrado: {'Sí ✓' if encontrado_astar else 'No'}")
print(f"  • Longitud: {len(plan_astar)} acciones")

if encontrado_astar:
    print(f"\n📝 PLAN:")
    for i, accion in enumerate(plan_astar, 1):
        print(f"  {i}. {accion}")

## Verificación del Plan

In [ ]:
print("\n" + "="*80)
print("VERIFICACIÓN DEL PLAN A*")
print("="*80)

if encontrado_astar and plan_astar:
    print(f"\n📊 Ejecución paso a paso del plan:")
    print(f"\nEstado inicial:")
    print(f"  • Robot en: Zona A")
    print(f"  • Caja en: Zona A")
    print(f"  • Carretilla en: Almacén")
    print(f"  • Zona C: Bloqueada")
    
    for idx, accion in enumerate(plan_astar, 1):
        print(f"\nPaso {idx}: {accion}")
    
    print(f"\n" + "="*80)
    print(f"✅ OBJETIVO ALCANZADO: Caja en Zona C")
    print(f"Plan óptimo con {len(plan_astar)} acciones")
    print("="*80)

## Comparación de Algoritmos

In [ ]:
print("\n" + "="*80)
print("COMPARACIÓN BFS vs A*")
print("="*80)

print(f"\n{'Métrica':<30} {'BFS':<25} {'A*':<25}")
print("-" * 80)
print(f"{'Tiempo (segundos)':<30} {tiempo_bfs:<25.6f} {tiempo_astar:<25.6f}")
print(f"{'Iteraciones':<30} {iter_bfs:<25} {iter_astar:<25}")
print(f"{'Longitud del plan':<30} {len(plan_bfs):<25} {len(plan_astar):<25}")
print(f"{'Optimalidad':<30} {'Garantizada':<25} {'Garantizada':<25}")

if iter_astar < iter_bfs:
    mejora = (iter_bfs - iter_astar) / iter_bfs * 100
    print(f"\n📊 A* es {mejora:.1f}% más eficiente que BFS")
    print(f"   (Explora {iter_bfs - iter_astar} menos nodos)")

## Análisis Crítico

### 1. ¿Es el plan resultante óptimo?

**Respuesta: SÍ, es óptimo.**

- **BFS garantiza optimalidad:** Encuentra el camino más corto (plan con menos acciones)
- **A* también es óptimo:** Con heurística admisible, garantiza encontrar la solución óptima
- **Verificación:** El plan de 8 acciones es el mínimo posible porque:
  - Necesita ir a almacén (1 acción) + tomar carretilla (1 acción) = 2 pasos
  - Necesita volver a A (1 acción) + recoger caja (1 acción) = 2 pasos
  - Necesita mover A→B→C (2 acciones) + despejar zona C (1 acción) = 3 pasos
  - Necesita depositar caja (1 acción)
  - **Total mínimo: 8 acciones**

### 2. ¿Qué sucede si invertimos el objetivo o cambiamos la zona ocupada?

- **Objetivo invertido (caja en A):** Ya se cumple en el estado inicial → plan vacío (0 acciones)
- **Zona B bloqueada:** El robot tendría que ir por almacén (ruta alternativa más larga)
- **Sin zona bloqueada:** El plan sería más corto (7 acciones, sin despejar)

### 3. ¿Qué ocurriría con duraciones, recursos limitados o incertidumbre sensorial?

- **Temporal Planning:** Cada acción tendría duración. Necesitaríamos minimizar makespan (tiempo total)
- **Recursos limitados:** Predicados de energía, batería. El planificador verificaría que no se agoten
- **Incertidumbre sensorial:** El robot no sabe su posición exacta. Cambiaría a Contingency Planning (políticas vs planes lineales)

### 4. ¿Qué planificador es más adecuado para escala real?

**Fast Forward (FF) es ideal para almacenes reales:**
- **BFS/A*:** Buenos para problemas pequeños pero no escalan
- **FF:** Usa heurística relajada, muy rápido en espacios grandes
- **GraphPlan:** Óptimo pero exponencial en espacio
- **SATplan:** Preciso pero overhead de traducción a SAT

**Conclusión:** Para un almacén con 100+ cajas y 50+ zonas, FF es superior (segundos vs minutos)